<a href="https://colab.research.google.com/github/lovnishverma/Python-Getting-Started/blob/main/Walrus_Operator_Masterclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://media.tenor.com/zcnBHya8CoMAAAAM/walrus-walrus-waving.gif" style="text-align:left;">

# 🦭 The Walrus Operator (`:=`) — A Complete Masterclass

> *"The best code is not just code that works — it's code that communicates."*

---

## 📋 Table of Contents

1. [What Is It & Why Does It Exist?](#section1)
2. [The Core Mechanic](#section2)
3. [Use Case 1 — `if` Statements](#section3)
4. [Use Case 2 — `while` Loops & Streams](#section4)
5. [Use Case 3 — List Comprehensions](#section5)
6. [Use Case 4 — Regular Expressions](#section6)
7. [Use Case 5 — Avoiding Redundant Function Calls](#section7)
8. [Use Case 6 — Nested Data Validation](#section8)
9. [Common Pitfalls & Anti-Patterns](#section9)
10. [The Golden Rules](#section10)
11. [Quick-Reference Cheat Sheet](#section11)
12. [Practice Exercises](#section12)

---

<a id='section1'></a>
## 1. 🧐 What Is It & Why Does It Exist?

The **walrus operator** (`:=`), officially called the **assignment expression**, was introduced in **Python 3.8** via [PEP 572](https://peps.python.org/pep-0572/).

Its nickname comes from how `:=` looks like a walrus lying on its side — the two dots are the eyes, the equals sign is the tusks. 🦭

### The Problem It Solves

Before `:=`, Python had a strict rule: **assignment (`=`) is a statement, not an expression**. This meant you could never assign and use a value *in the same breath*. You always needed two lines.

The walrus operator breaks this rule intentionally and carefully. It lets you **assign a value to a variable AND return that value simultaneously**, so you can use the result immediately without a separate line.

### The Fundamental Difference

```
Regular assignment  =    →  Statement only (cannot be used inside expressions)
Walrus operator     :=   →  Expression (assigns AND returns the value)
```

Think of `:=` as the answer to: *"Can I assign this AND use it at the same time?"* — now yes.

<a id='section2'></a>
## 2. ⚙️ The Core Mechanic

Let's strip everything away and understand the bare mechanism.

In [73]:
# ── THE CORE MECHANIC ──────────────────────────────────────────────────────────
#
# (variable := expression) does TWO things:
#   1. Evaluates the right-hand expression
#   2. Assigns the result to 'variable'
#   3. Returns that result so it can be used in the outer expression
#
# Without walrus: two steps, two lines
x = 10
print(x)          # 10

# With walrus: both steps in one expression
print(y := 10)    # 10  ← assigns 10 to y AND prints it
print(y)          # 10  ← y is now accessible here too!

print("\n--- The walrus operator returns the value it assigns ---")
result = (z := 42) * 2   # z gets 42, and (z := 42) evaluates to 42
print(f"z = {z}, result = {result}")  # z = 42, result = 84

10
10
10

--- The walrus operator returns the value it assigns ---
z = 42, result = 84


In [74]:
# ── PARENTHESES ARE REQUIRED (almost always) ──────────────────────────────────
#
# The walrus operator has very low precedence, so you usually need parentheses.
# Think of the parens as saying: "evaluate this assignment AS an expression"

numbers = [3, 7, 2, 9, 1]

# ✅ Correct — parentheses make precedence explicit
if (n := len(numbers)) > 4:
    print(f"List has {n} elements — that's more than 4!")

# ❌ This would be a SyntaxError:
# if n := len(numbers) > 4:   ← Python parses this as n := (len(numbers) > 4)
#     ...                        which assigns True/False, not the length!

List has 5 elements — that's more than 4!


<a id='section3'></a>
## 3. 🔀 Use Case 1 — `if` Statements

The simplest and most common use: **compute a value, check it, and use it** — all in one place.

In [75]:
# ── SCENARIO: Validate and process user input ──────────────────────────────────

import re

def get_username():
    """Simulates fetching a username from a database or input."""
    return "  alice_2024  "  # note: has leading/trailing whitespace

# ❌ OLD WAY — 3 separate lines to compute, check, and use
print("=== Old Way ===")
username = get_username()
stripped = username.strip()
if stripped:
    print(f"Welcome, {stripped}!")

# ✅ WALRUS WAY — check and use in one expression
print("\n=== Walrus Way ===")
if cleaned := get_username().strip():
    print(f"Welcome, {cleaned}!")

=== Old Way ===
Welcome, alice_2024!

=== Walrus Way ===
Welcome, alice_2024!


In [76]:
# ── SCENARIO: if / elif / else chain where each branch does its own calculation ─

def compute_score(data):
    """A somewhat expensive function we want to call only once per check."""
    return sum(data) / len(data) if data else 0

samples = [72, 85, 91, 68, 77]

# ❌ OLD WAY — compute_score() is called EVERY time we enter a new branch
print("=== Old Way ===")
score = compute_score(samples)
if score >= 90:
    print(f"Score {score:.1f}: Grade A")
elif score >= 75:
    print(f"Score {score:.1f}: Grade B")
else:
    print(f"Score {score:.1f}: Grade C")

# ✅ WALRUS WAY — clean and minimal
print("\n=== Walrus Way ===")
if (score := compute_score(samples)) >= 90:
    print(f"Score {score:.1f}: Grade A")
elif score >= 75:
    print(f"Score {score:.1f}: Grade B")  # 'score' is still in scope!
else:
    print(f"Score {score:.1f}: Grade C")

=== Old Way ===
Score 78.6: Grade B

=== Walrus Way ===
Score 78.6: Grade B


### 💡 Key Insight

Notice that in the walrus version, `score` is assigned in the `if` line and is **still accessible** in the `elif` and `else` blocks. The variable is bound to the enclosing scope — not just the `if` block. This is by design.

<a id='section4'></a>
## 4. 🔁 Use Case 2 — `while` Loops & Data Streams

This is where the walrus operator is arguably **most powerful**. The classic `while` loop pattern for reading data has always been awkward. The walrus operator fixes it elegantly.

In [77]:
import io

# ── SCENARIO: Reading a file in chunks ────────────────────────────────────────

mock_file_content = "AAABBBCCCDDDEEEFFFGGGHHH"

# ❌ OLD WAY — The 'Loop-and-a-Half' anti-pattern
# You have to read once before the loop, and again at the end of the loop body.
# This duplicates the read call and is easy to mess up.
print("=== Old Way (loop-and-a-half) ===")
mock_file = io.StringIO(mock_file_content)
chunk = mock_file.read(3)        # ← Read #1 (outside loop)
while chunk:
    print(f"  Processing: {chunk}")
    chunk = mock_file.read(3)    # ← Read #2 (inside loop) — duplicated logic!

# ✅ WALRUS WAY — single, clean, idiomatic
print("\n=== Walrus Way ===")
mock_file = io.StringIO(mock_file_content)
while chunk := mock_file.read(3):   # Read, assign, AND check — all in one!
    print(f"  Processing: {chunk}")

=== Old Way (loop-and-a-half) ===
  Processing: AAA
  Processing: BBB
  Processing: CCC
  Processing: DDD
  Processing: EEE
  Processing: FFF
  Processing: GGG
  Processing: HHH

=== Walrus Way ===
  Processing: AAA
  Processing: BBB
  Processing: CCC
  Processing: DDD
  Processing: EEE
  Processing: FFF
  Processing: GGG
  Processing: HHH


In [78]:
# ── SCENARIO: Interactive input loop ──────────────────────────────────────────
# (We'll simulate user input with a queue instead of actual input())

from collections import deque

simulated_inputs = deque(["hello", "world", "quit", "this won't be reached"])

def simulated_input(prompt):
    val = simulated_inputs.popleft() if simulated_inputs else "quit"
    print(f"{prompt}{val}")  # show what the 'user' typed
    return val

# ❌ OLD WAY
print("=== Old Way ===")
simulated_inputs = deque(["hello", "world", "quit", "this won't be reached"])
command = simulated_input("> ")
while command != "quit":
    print(f"  Echo: {command}")
    command = simulated_input("> ")
print("Goodbye!")

# ✅ WALRUS WAY
print("\n=== Walrus Way ===")
simulated_inputs = deque(["hello", "world", "quit", "this won't be reached"])
while (command := simulated_input("> ")) != "quit":
    print(f"  Echo: {command}")
print("Goodbye!")

=== Old Way ===
> hello
  Echo: hello
> world
  Echo: world
> quit
Goodbye!

=== Walrus Way ===
> hello
  Echo: hello
> world
  Echo: world
> quit
Goodbye!


In [79]:
# ── SCENARIO: Network-style data accumulation ──────────────────────────────────
# Simulate receiving packets until we get None (connection closed)

import random

def receive_packet(packet_queue):
    """Simulates receiving a packet. Returns None when stream ends."""
    return packet_queue.pop(0) if packet_queue else None

packet_stream = [b"PKT_001", b"PKT_002", b"PKT_003", b"PKT_004", None]
received = []

# ✅ WALRUS WAY — None is falsy, so the while loop stops automatically
while packet := receive_packet(packet_stream):
    received.append(packet)
    print(f"  Received: {packet}")

print(f"\nTotal packets received: {len(received)}")

  Received: b'PKT_001'
  Received: b'PKT_002'
  Received: b'PKT_003'
  Received: b'PKT_004'

Total packets received: 4


<a id='section5'></a>
## 5. 📋 Use Case 3 — List Comprehensions

This is the most **performance-critical** use of the walrus operator. It lets you call a function once and use its result in both the filter and the output of a comprehension.

In [80]:
import time

# Simulate an expensive transformation (e.g., a neural net forward pass)
call_count = 0

def expensive_transform(x):
    global call_count
    call_count += 1
    return x ** 2 - x + 1  # Some non-trivial calculation

data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
threshold = 20

# ❌ OLD WAY — expensive_transform() is called TWICE per item!
call_count = 0
old_results = [expensive_transform(x) for x in data if expensive_transform(x) > threshold]
print(f"Old way: {old_results}")
print(f"  → Function was called {call_count} times for {len(data)} items!")

# ✅ WALRUS WAY — function is called ONCE per item
call_count = 0
new_results = [result for x in data if (result := expensive_transform(x)) > threshold]
print(f"\nWalrus way: {new_results}")
print(f"  → Function was called {call_count} times for {len(data)} items!")
print(f"  → Saved {len(data)} extra function calls! 🚀")

Old way: [21, 31, 43, 57, 73, 91]
  → Function was called 16 times for 10 items!

Walrus way: [21, 31, 43, 57, 73, 91]
  → Function was called 10 times for 10 items!
  → Saved 10 extra function calls! 🚀


In [81]:
# ── SCENARIO: Filter and transform a dataset in one pass ──────────────────────
# Common in ML data preprocessing pipelines

import math

raw_sensor_readings = [-5.2, 0.0, 3.1, 12.8, -1.0, 7.4, 22.3, 0.5, 11.1]

def normalize(x, scale=10.0):
    """Normalize positive readings to [0, 1] range. Rejects negatives."""
    if x <= 0:
        return None  # Invalid reading
    return min(x / scale, 1.0)

# ✅ Filter out None (invalid) AND collect normalized values in ONE comprehension
valid_readings = [
    normed
    for x in raw_sensor_readings
    if (normed := normalize(x)) is not None
]

print(f"Raw readings:    {raw_sensor_readings}")
print(f"Valid & normed:  {[round(v, 3) for v in valid_readings]}")
print(f"Dropped {len(raw_sensor_readings) - len(valid_readings)} invalid readings")

Raw readings:    [-5.2, 0.0, 3.1, 12.8, -1.0, 7.4, 22.3, 0.5, 11.1]
Valid & normed:  [0.31, 1.0, 0.74, 1.0, 0.05, 1.0]
Dropped 3 invalid readings


In [82]:
# ── DICT COMPREHENSION with walrus ────────────────────────────────────────────
# You can use := in dict comprehensions too!

words = ["python", "walrus", "operator", "is", "cool", "and", "useful"]

# Build a dict of {word: length} but ONLY for words longer than 3 characters
long_words = {
    word: length
    for word in words
    if (length := len(word)) > 3
}

print("Words longer than 3 chars with their lengths:")
for word, length in long_words.items():
    print(f"  {word!r:12} → {length} chars")

Words longer than 3 chars with their lengths:
  'python'     → 6 chars
  'walrus'     → 6 chars
  'operator'   → 8 chars
  'cool'       → 4 chars
  'useful'     → 6 chars


<a id='section6'></a>
## 6. 🔍 Use Case 4 — Regular Expressions

Regex is a **perfect** fit for the walrus operator. The classic pattern is: search for a match, and if it exists, do something with it. Without walrus, you always needed an intermediate variable.

In [83]:
import re

log_entries = [
    "2024-01-15 ERROR: Disk quota exceeded (used: 98%)",
    "2024-01-15 INFO: Backup completed successfully",
    "2024-01-16 ERROR: Connection timeout after 30s",
    "2024-01-16 WARNING: Memory usage at 85%",
    "2024-01-17 ERROR: Null pointer dereference in module auth",
]

error_pattern = re.compile(r"(\d{4}-\d{2}-\d{2}) ERROR: (.+)")

# ❌ OLD WAY — boilerplate match variable for every entry
print("=== Old Way ===")
errors_found = []
for entry in log_entries:
    match = error_pattern.search(entry)  # must define 'match' first
    if match:
        errors_found.append({"date": match.group(1), "message": match.group(2)})

# ✅ WALRUS WAY — match and use in one line
print("\n=== Walrus Way ===")
errors_found = [
    {"date": m.group(1), "message": m.group(2)}
    for entry in log_entries
    if (m := error_pattern.search(entry))  # assign AND test truthiness
]

print(f"Found {len(errors_found)} errors:")
for err in errors_found:
    print(f"  [{err['date']}] {err['message']}")

=== Old Way ===

=== Walrus Way ===
Found 3 errors:
  [2024-01-15] Disk quota exceeded (used: 98%)
  [2024-01-16] Connection timeout after 30s
  [2024-01-17] Null pointer dereference in module auth


In [84]:
# ── SCENARIO: Try multiple patterns, stop on first match ──────────────────────

def parse_contact(text):
    """Parse a contact string that might be a phone, email, or username."""
    if m := re.match(r"^\+?1?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}$", text):
        return f"📱 Phone number: {m.group(0)}"
    elif m := re.match(r"^[\w.+-]+@[\w-]+\.[\w.]+$", text):
        return f"📧 Email: {m.group(0)}"
    elif m := re.match(r"^@([\w]+)$", text):
        return f"🐦 Username: @{m.group(1)}"
    else:
        return f"❓ Unknown format: {text}"

contacts = ["555-867-5309", "alice@example.com", "@walrus_fan", "not a contact"]
for contact in contacts:
    print(parse_contact(contact))

📱 Phone number: 555-867-5309
📧 Email: alice@example.com
🐦 Username: @walrus_fan
❓ Unknown format: not a contact


<a id='section7'></a>
## 7. ⚡ Use Case 5 — Avoiding Redundant Function Calls

When you need to call the same function multiple times in close proximity, the walrus operator lets you cache the result cleanly.

In [85]:
# ── SCENARIO: Conditional logic that uses the same computed value repeatedly ───

def fetch_api_data(endpoint):
    """Simulates an API call that returns data or None."""
    fake_db = {
        "/users/42": {"name": "Alice", "role": "admin", "active": True},
        "/users/99": None,  # user not found
    }
    return fake_db.get(endpoint)

endpoints_to_check = ["/users/42", "/users/99"]

for endpoint in endpoints_to_check:
    print(f"\nChecking {endpoint}:")

    # ✅ WALRUS WAY — fetch once, use multiple times
    if data := fetch_api_data(endpoint):
        print(f"  ✅ Found user: {data['name']}")
        print(f"  Role: {data['role']}")
        print(f"  Active: {data['active']}")
    else:
        print(f"  ❌ No data returned for {endpoint}")


Checking /users/42:
  ✅ Found user: Alice
  Role: admin
  Active: True

Checking /users/99:
  ❌ No data returned for /users/99


In [86]:
# ── SCENARIO: Chained operations — walrus in an assert statement ───────────────
# Useful for tests and data validation pipelines

def load_config(path):
    """Simulates loading a config file."""
    return {"model": "gpt", "lr": 0.001, "epochs": 10, "batch_size": 32}

# You can assert and capture in one shot (great for quick sanity checks):
assert (config := load_config("config.yaml")) is not None, "Config failed to load!"

# Now 'config' is guaranteed to be valid and ready to use
print(f"Config loaded: {config}")
print(f"Training for {config['epochs']} epochs at lr={config['lr']}")

Config loaded: {'model': 'gpt', 'lr': 0.001, 'epochs': 10, 'batch_size': 32}
Training for 10 epochs at lr=0.001


<a id='section8'></a>
## 8. 🗂️ Use Case 6 — Nested Data Validation

When working with deeply nested dictionaries or JSON (common in APIs and configs), walrus lets you safely drill down without a cascade of intermediate variables.

In [87]:
# ── SCENARIO: Safely extract values from a nested JSON response ───────────────

api_response = {
    "status": "ok",
    "payload": {
        "user": {
            "profile": {
                "address": {
                    "city": "San Francisco",
                    "country": "USA"
                }
            }
        }
    }
}

# ❌ OLD WAY — tedious and verbose
payload = api_response.get("payload")
if payload:
    user = payload.get("user")
    if user:
        profile = user.get("profile")
        if profile:
            address = profile.get("address")
            if address:
                print(f"City (old way): {address.get('city')}")

# ✅ WALRUS WAY — cleaner, each step is one line
if (payload := api_response.get("payload")) \
        and (user := payload.get("user")) \
        and (profile := user.get("profile")) \
        and (address := profile.get("address")):
    print(f"City (walrus):  {address.get('city')}")

City (old way): San Francisco
City (walrus):  San Francisco


In [88]:
# ── SCENARIO: Batch processing with early termination ─────────────────────────

from typing import Optional

def validate_email(email: str) -> Optional[str]:
    """Returns the domain if email is valid, else None."""
    pattern = r"^[\w.+-]+@([\w-]+\.[\w.]+)$"
    if m := re.match(pattern, email):
        return m.group(1)
    return None

submissions = [
    "alice@gmail.com",
    "not_an_email",
    "bob@company.org",
    "bad@",
    "carol@university.edu"
]

# Process valid emails, capture the domain in the same check
domain_stats = {}
for email in submissions:
    if domain := validate_email(email):
        domain_stats[domain] = domain_stats.get(domain, 0) + 1
        print(f"✅ Valid:   {email:30} → domain: {domain}")
    else:
        print(f"❌ Invalid: {email}")

print(f"\nDomain stats: {domain_stats}")

✅ Valid:   alice@gmail.com                → domain: gmail.com
❌ Invalid: not_an_email
✅ Valid:   bob@company.org                → domain: company.org
❌ Invalid: bad@
✅ Valid:   carol@university.edu           → domain: university.edu

Domain stats: {'gmail.com': 1, 'company.org': 1, 'university.edu': 1}


<a id='section9'></a>
## 9. ⚠️ Common Pitfalls & Anti-Patterns

The walrus operator can hurt readability when misused. Here are the patterns to avoid.

In [89]:
# ── PITFALL 1: Walrus when a plain assignment is clearer ──────────────────────

data = [1, 2, 3, 4, 5]

# ❌ Anti-pattern — using walrus at the top level of an expression (just use =)
# This is valid Python but confusing — just use regular assignment
# total := sum(data)   ← SyntaxError at top level anyway!

# ❌ Confusing in a simple assignment context
x = (y := 10) + 5  # Why use walrus here? Just write x = 15, y = 10 separately

# ✅ Only use walrus when it REMOVES a separate line that serves no other purpose
print("Walrus is best when it eliminates a throwaway pre-assignment step.")

Walrus is best when it eliminates a throwaway pre-assignment step.


In [90]:
# ── PITFALL 2: Nesting walrus operators ───────────────────────────────────────

# ❌ Anti-pattern — nested walrus is almost always unreadable
data = [1, 4, 9, 16, 25, 36]

# This is technically correct but nightmarish to read:
result = [b for x in data if (a := x * 2) > 10 if (b := a + 1) < 50]
print(f"Nested walrus result: {result}")  # Works, but don't do this!

# ✅ Better — just use a regular loop or break it apart
result_clear = []
for x in data:
    doubled = x * 2
    if doubled > 10:
        incremented = doubled + 1
        if incremented < 50:
            result_clear.append(incremented)
print(f"Clear version result: {result_clear}")

Nested walrus result: [19, 33]
Clear version result: [19, 33]


In [91]:
# ── PITFALL 3: Scope confusion — walrus leaks out of comprehensions! ──────────
#
# Regular comprehension variables are LOCAL to the comprehension.
# Walrus variables LEAK out to the enclosing scope. This can surprise you.

# Regular comprehension variable — does NOT leak
squares = [x**2 for x in range(5)]
# print(x)  # ← NameError! 'x' doesn't exist outside the comprehension

# Walrus variable — DOES leak
results = [y for x in range(5) if (y := x**2) > 4]
print(f"results: {results}")
print(f"y after comprehension: {y}")  # ← y is 16 (last assigned value)

print("\n⚠️  Be aware that walrus variables persist after the comprehension ends!")

results: [9, 16]
y after comprehension: 16

⚠️  Be aware that walrus variables persist after the comprehension ends!


In [92]:
# ── PITFALL 4: Using walrus in class or module scope ──────────────────────────
#
# Walrus cannot be used at the TOP LEVEL of a class body or module.
# It only works inside expressions (if, while, comprehensions, function calls).

# ❌ This would be a SyntaxError:
# class MyClass:
#     x := 10  ← SyntaxError!

# ✅ But this is fine (inside a method or function):
class DataProcessor:
    def process(self, items):
        if (total := sum(items)) > 100:
            return f"Large batch: {total} total"
        return f"Normal batch: {total} total"

dp = DataProcessor()
print(dp.process([10, 20, 30]))
print(dp.process([50, 60, 70]))

Normal batch: 60 total
Large batch: 180 total


<a id='section10'></a>
## 10. 📏 The Golden Rules

Before using `:=`, ask yourself these questions:

| # | Question | If YES → |
|---|----------|----------|
| 1 | Does it **eliminate a separate assignment line** with no other purpose? | ✅ Good candidate |
| 2 | Does it **avoid calling a function twice**? | ✅ Good candidate |
| 3 | Is the variable used **immediately** in the same expression? | ✅ Good candidate |
| 4 | Does it require me to **nest** multiple walrus operators? | ❌ Don't use it |
| 5 | Does it make the line **significantly harder to read at a glance**? | ❌ Don't use it |
| 6 | Could I use a regular `=` on the line just above? | ❌ Just use `=` |

### The One-Sentence Rule

> **Use `:=` only when it removes a line that exists solely to be checked or used in the very next line.**

The walrus operator is a *precision tool*, not a style preference. Use it surgically.

<a id='section11'></a>
## 11. 📄 Quick-Reference Cheat Sheet

In [105]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║              WALRUS OPERATOR (:=) — QUICK REFERENCE CHEAT SHEET            ║
# ╠══════════════════════════════════════════════════════════════════════════════╣
# ║  Introduced: Python 3.8 (PEP 572)                                          ║
# ║  Also known as: assignment expression, named expression                    ║
# ║  Nickname: walrus operator (from the :=  eyes and tusks emoji: 🦭)         ║
# ╠═══════════════════════════════╦══════════════════════════════════════════════╣
# ║  PATTERN                      ║  EXAMPLE                                    ║
# ╠═══════════════════════════════╬══════════════════════════════════════════════╣
# ║  if statement                 ║  if (n := len(data)) > 10: ...              ║
# ║  elif chain                   ║  elif (val := compute()) < 0: ...           ║
# ║  while loop (stream)          ║  while chunk := f.read(1024): ...           ║
# ║  while with sentinel          ║  while (line := input()) != 'quit': ...     ║
# ║  list comprehension           ║  [y for x in data if (y := f(x)) > 0]      ║
# ║  dict comprehension           ║  {w: n for w in words if (n:=len(w))>3}    ║
# ║  regex match                  ║  if m := re.search(pat, text): ...          ║
# ║  nested dict access           ║  if (a := d.get('k')) and (b := a.get(...)  ║
# ╠═══════════════════════════════╬══════════════════════════════════════════════╣
# ║  RULES & GOTCHAS              ║                                              ║
# ╠═══════════════════════════════╬══════════════════════════════════════════════╣
# ║  Parentheses needed           ║  if (n := len(x)) > 5, NOT if n:=len(x)>5  ║
# ║  Scope: leaks from compr.     ║  Variable lives in enclosing scope           ║
# ║  Cannot use at top of class   ║  Only inside expressions/functions          ║
# ║  Prefer = when not in expr.   ║  n = len(x); if n > 5: ... is often fine   ║
# ╚═══════════════════════════════╩══════════════════════════════════════════════╝

print("Cheat sheet above — keep this as reference!")

Cheat sheet above — keep this as reference!


<a id='section12'></a>
## 12. 🏋️ Practice Exercises

Test your understanding! Each exercise has a stub — write your solution using the walrus operator.

---

In [94]:
# ── EXERCISE 1 — Easy ─────────────────────────────────────────────────────────
#
# Rewrite the code below using the walrus operator.
# It should compute the average, check if it passes the threshold,
# and print it — all in a single if statement.

scores = [88, 92, 76, 95, 83]
threshold = 85

# --- Before (your reference) ---
avg = sum(scores) / len(scores)
if avg >= threshold:
    print(f"[Before] Class average {avg:.1f} passed the threshold!")

# --- Your task: rewrite with := ---
# YOUR CODE HERE
# if (avg := ...) >= threshold:
#     print(...)

[Before] Class average 86.8 passed the threshold!


In [95]:
# ── EXERCISE 2 — Medium ───────────────────────────────────────────────────────
#
# Rewrite the list comprehension below using walrus to avoid calling
# `classify_word()` twice per item.

def classify_word(word):
    """Returns 'short', 'medium', or 'long' based on word length."""
    l = len(word)
    if l <= 3: return 'short'
    elif l <= 6: return 'medium'
    else: return 'long'

vocab = ["cat", "python", "is", "magnificent", "and", "powerful"]

# --- Before (calls classify_word TWICE per item!) ---
long_words_old = [classify_word(w) for w in vocab if classify_word(w) == 'long']
print(f"Long word classifications (old): {long_words_old}")

# --- Your task: rewrite with := (call classify_word only ONCE per item) ---
# long_words_new = [... for w in vocab if (cat := ...) == 'long']
# YOUR CODE HERE

Long word classifications (old): ['long', 'long']


In [96]:
# ── EXERCISE 3 — Hard ─────────────────────────────────────────────────────────
#
# You're building a simple CSV parser. Write a while loop that:
#   1. Reads one line at a time from `csv_stream`
#   2. Strips whitespace and skips blank lines
#   3. Splits by comma and prints the parsed row
#
# Use the walrus operator in the while condition.

import io

csv_data = """name,age,city
Alice,30,New York

Bob,25,London
Carol,35,Tokyo
"""

csv_stream = io.StringIO(csv_data)

# YOUR CODE HERE
# while (line := ...)...:
#     if not line: continue
#     row = ...
#     print(row)

In [97]:
# ── EXERCISE SOLUTIONS ────────────────────────────────────────────────────────
# Uncomment to check your work!

# Solution 1:
# if (avg := sum(scores) / len(scores)) >= threshold:
#     print(f"[Walrus] Class average {avg:.1f} passed the threshold!")

# Solution 2:
# long_words_new = [cat for w in vocab if (cat := classify_word(w)) == 'long']
# print(f"Long word classifications (walrus): {long_words_new}")

# Solution 3:
# csv_stream = io.StringIO(csv_data)  # reset stream
# while line := csv_stream.readline():
#     if not (line := line.strip()):
#         continue
#     row = line.split(',')
#     print(row)

print("Uncomment the solution you want to check above!")

Uncomment the solution you want to check above!


---

## 🎓 Summary

You've completed the Walrus Operator Masterclass! Here's what you learned:

- **What it is**: An assignment expression (`:=`) that assigns AND returns a value simultaneously.
- **When to use it**: `if` statements, `while` loops with streams, list/dict comprehensions, and regex checks.
- **Why it helps**: Eliminates redundant assignments, prevents double function calls, and removes the "loop-and-a-half" pattern.
- **When NOT to use it**: When nesting, when a plain `=` would do, or when it hurts readability.
- **Key gotcha**: Variables assigned with `:=` inside comprehensions **leak into the enclosing scope**.

### Further Reading

- 📖 [PEP 572 — Assignment Expressions](https://peps.python.org/pep-0572/) — the original proposal with full rationale
- 📖 [Python 3.8 What's New](https://docs.python.org/3/whatsnew/3.8.html#assignment-expressions) — official docs
- 📖 [Real Python — Walrus Operator](https://realpython.com/python-walrus-operator/) — in-depth tutorial

---
*Happy coding! 🦭*

# Practicals

**The Walrus Operator**

Python 3.8 introduced the := operator, known as the "walrus operator". It assigns values to variables as part of a larger expression:

In [98]:
name = "Lovnish"

print(name)

Lovnish


In [99]:
# print(name = "Lovnish") #will throw a error

In [100]:
print(name := "Lovnish")

Lovnish


In [104]:
data = [1, 4, 9, 16, 25, 36, 45,75,85,96,85,85,2,6,35,6,5,63]

print("Without Walrus Operator")

n = len(data)
if (n) > 10:
    print(f"Too long ({n} items)")

print("===================")
print("With Walrus Operator")
if (x := len(data)) > 10:
    print(f"Too long ({x} items)")

Without Walrus Operator
Too long (18 items)
With Walrus Operator
Too long (18 items)


In [107]:
print("\n=== old Way ===")
cleaned = get_username().strip()
if cleaned:
    print(f"Welcome, {cleaned}!")


=== Walrus Way ===
Welcome, alice_2024!


In [108]:
print("\n=== Walrus Way ===")
if cleaned := get_username().strip():
    print(f"Welcome, {cleaned}!")


=== Walrus Way ===
Welcome, alice_2024!
